In [2]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [3]:
import os

database_path = os.path.abspath('data/crash_data.db')
print(f"Absolute database path: {database_path}")

if not os.path.exists(database_path):
    print(f"Database file not found at {database_path}")
else:
    print(f"Database file found at {database_path}")


Absolute database path: /Users/terid/CodeYou_Capstone/data/crash_data.db
Database file found at /Users/terid/CodeYou_Capstone/data/crash_data.db


In [4]:
# Database path
database_path = 'data/crash_data.db'

# Execute the query and fetch the data into a pandas DataFrame
query = """
SELECT strftime('%Y', CollisionDate) AS Year,
       strftime('%m', CollisionDate) AS Month,
       SUM(NumberKilled) AS TotalDeaths
  FROM ksp_incidents
 WHERE NumberKilled > 0
 GROUP BY Year, Month
 ORDER BY Year, Month;
"""

# Execute the query and load the data into a pandas DataFrame
with sqlite3.connect(database_path) as conn:
    df = pd.read_sql_query(query, conn)

# Combine Year and Month into a single datetime column for Plotly
df['Date'] = pd.to_datetime(df['Year'] + '-' + df['Month'] + '-01')


In [5]:
# Create the line graph
fig = px.line(df, x='Date', y='TotalDeaths', title='Total Deaths by Year and Month',
              labels={'TotalDeaths': 'Number of Deaths', 'Date': 'Date'})

# Update x-axis to show only quarterly months
fig.update_xaxes(
    dtick="M3",
    tickformat="%b %Y",
    ticklabelmode="period"
)

# Show the figure
fig.show()


In [6]:
# Create the bar chart
fig = px.bar(df, x='Date', y='TotalDeaths', title='Total Deaths by Year and Month',
             labels={'TotalDeaths': 'Number of Deaths', 'Date': 'Date'})

# Update x-axis to show only quarterly months
fig.update_xaxes(
    dtick="M3",
    tickformat="%b %Y",
    ticklabelmode="period"
)

# Show the figure
fig.show()


In [7]:
# Execute the query and fetch the data into a pandas DataFrame
query = """
SELECT strftime('%Y', CollisionDate) AS Year,
       strftime('%m', CollisionDate) AS Month,
       SUM(NumberKilled) AS TotalDeaths,
       SUM(NumberInjured) AS TotalInjuries
  FROM ksp_incidents
 WHERE NumberKilled > 0 OR NumberInjured > 0
 GROUP BY Year, Month
 ORDER BY Year, Month;
"""
df = pd.read_sql_query(query, conn)

# Combine Year and Month into a single datetime column for Plotly
df['Date'] = pd.to_datetime(df['Year'] + '-' + df['Month'] + '-01')

In [8]:
# Melt the dataframe to have a long format suitable for Plotly
df_melted = df.melt(id_vars=['Date'], value_vars=['TotalDeaths', 'TotalInjuries'],
                    var_name='Type', value_name='Count')

In [9]:
# Create the bar chart
fig = px.bar(df_melted, x='Date', y='Count', color='Type', barmode='group',
             title='Total Deaths and Injuries by Year and Month',
             labels={'Count': 'Number of People', 'Date': 'Date'})

# Update x-axis to show only quarterly months
fig.update_xaxes(
    dtick="M3",
    tickformat="%b %Y",
    ticklabelmode="period"
)

# Show the figure
fig.show()


In [10]:
# Execute the query and fetch the data into a pandas DataFrame
query = """
SELECT strftime('%Y', k.CollisionDate) AS Year,
       strftime('%m', k.CollisionDate) AS Month,
       p.personTypecde AS PersonType,
       SUM(k.NumberKilled) AS TotalDeaths,
       SUM(k.NumberInjured) AS TotalInjuries
  FROM ksp_incidents k
  JOIN ksp_person p ON k.IncidentID = p.IncidentID
 WHERE k.NumberKilled > 0 OR k.NumberInjured > 0
 GROUP BY Year, Month, PersonType
 ORDER BY Year, Month, PersonType;
"""
df = pd.read_sql_query(query, conn)

# Combine Year and Month into a single datetime column for Plotly
df['Date'] = pd.to_datetime(df['Year'] + '-' + df['Month'] + '-01')


In [11]:
# Melt the dataframe to have a long format suitable for Plotly
df_melted = df.melt(id_vars=['Date', 'PersonType'], value_vars=['TotalDeaths', 'TotalInjuries'],
                    var_name='Type', value_name='Count')

In [12]:
# Create the bar chart
fig = px.bar(df_melted, x='Date', y='Count', color='Type', barmode='group',
             facet_col='PersonType', title='Total Deaths and Injuries by Year, Month, and Person Type',
             labels={'Count': 'Number of People', 'Date': 'Date', 'PersonType': 'Person Type'})

# Update x-axis to show only quarterly months
fig.update_xaxes(
    dtick="M3",
    tickformat="%b %Y",
    ticklabelmode="period"
)

# Show the figure
fig.show()


In [13]:
# Execute the query and fetch the data into a pandas DataFrame
query = """
SELECT strftime('%Y', k.CollisionDate) AS Year,
       p.personTypecde AS PersonType,
       SUM(k.NumberKilled) AS TotalDeaths
  FROM ksp_incidents k
  JOIN ksp_person p ON k.IncidentID = p.IncidentID
 WHERE k.NumberKilled > 0
 GROUP BY Year, PersonType
 ORDER BY Year, PersonType;
"""
df = pd.read_sql_query(query, conn)


In [14]:
# Map the person type codes to their respective labels
person_type_labels = {
    '1': 'Driver',
    '2': 'Passenger',
    '3': 'Pedestrian',
    '4': 'Animal-Drawn/Ridden',
    '5': 'Bicyclist',
    '6': 'Train Engineer',
    '7': 'Witness',
    '8': 'Owner',
    '9': 'Property Damage Owner'
}

df['PersonType'] = df['PersonType'].map(person_type_labels)

In [15]:
# Create the pie chart
fig = px.pie(df, values='TotalDeaths', names='PersonType',
             title='Total Deaths Categorized by Person Type 2020 - 2024',
             labels={'TotalDeaths': 'Number of Deaths', 'PersonType': 'Person Type'})

# Show the figure
fig.show()


In [16]:
# Execute the query and fetch the data into a pandas DataFrame
query = """
SELECT DISTINCT(i.incidentid), strftime('%Y', i.CollisionDate) AS Year,
       r.Route_Type, r.Speed_Limit_Posted_MPH as Posted_Speed_Limit
  FROM ksp_incidents as i
  JOIN Roadway_Characteristics_API as r
  JOIN ksp_factors as f
    ON i.IncidentID = r.IncidentID
 WHERE f.Factor = 7 AND r.Route_type IN ('I', 'PKWY', 'US', 'KY')
 GROUP BY i.incidentid, Year, r.Route_Type;
"""
df = pd.read_sql_query(query, conn)

Out of 4105 incidents, 392 are designated as non-state-maintained routes (interstates, parkways, US routes and KY routes). These have been eliminated from the following analysis.

In [17]:
# Create a count of incidents by year and route type
df_count = df.groupby(['Year', 'Route_Type']).size().reset_index(name='IncidentCount')


# Specify the order of route types
category_order = ['I', 'PKWY', 'US', 'KY']

# Create the bar chart
fig = px.bar(df_count, x='Year', y='IncidentCount', color='Route_Type', barmode='group',
             title='Incidents by Year and Route Type Involving Excessive Speed as a Factor',
             labels={'IncidentCount': 'Number of Incidents', 'Year': 'Year', 'Route_Type': 'Route Type'},
             category_orders={'Route_Type': category_order})
# Show the figure
fig.show()